In [103]:
%reload_ext autoreload
%autoreload 2

from src.generate_surface_code import SurfaceCode
from src.TN_decoder import decoder
from src.parse_syndrome import *
import stim
from matplotlib import pyplot as plt
import numpy as np
from tqdm import tqdm
import pymatching

In [104]:
distance = 3
noise_model = "depolarize"
noise = 0.1
chi = 8
nshots = 1000

In [105]:
def count_logical_errors(code, num_shots: int) -> int:

    circuit = code.circuit
    sampler = circuit.compile_detector_sampler()
    detection_events, observable_flips = sampler.sample(shots = num_shots, separate_observables=True)
    predictions = []

    for event in tqdm(detection_events):
        predictions.append(decoder(code, event, chi))


    fails = sum([1 if not predictions[j] == observable_flips[j] else 0 for j in range(num_shots)])
    return fails

In [106]:
def count_MWPM_logical_errors(circuit: stim.Circuit, num_shots: int) -> int:
    # Sample the circuit.
    sampler = circuit.compile_detector_sampler()
    detection_events, observable_flips = sampler.sample(num_shots, separate_observables=True)

    # Configure a decoder using the circuit.
    detector_error_model = circuit.detector_error_model(decompose_errors=True)
    matcher = pymatching.Matching.from_detector_error_model(detector_error_model)

    # Run the decoder.
    predictions = matcher.decode_batch(detection_events)

    # Count the mistakes.
    num_errors = 0
    for shot in range(num_shots):
        actual_for_shot = observable_flips[shot]
        predicted_for_shot = predictions[shot]
        if not np.array_equal(actual_for_shot, predicted_for_shot):
            num_errors += 1
    return num_errors

In [107]:
code = SurfaceCode(distance, noise_model, noise)

In [108]:
code.dem

stim.DetectorErrorModel('''
    error(0.0345253) D0
    error(0.0345253) D0 D1
    error(0.0345253) D0 D1 ^ D7 L0
    error(0.0345253) D0 D2
    error(0.0345253) D0 D2 ^ D6 D7
    error(0.0345253) D0 ^ D6 L0
    error(0.0345253) D1
    error(0.0345253) D1 D3
    error(0.0345253) D1 D3 ^ D7 D8
    error(0.0345253) D1 ^ D8 L0
    error(0.0345253) D2
    error(0.0345253) D2 D3
    error(0.0345253) D2 D3 ^ D7 D10
    error(0.0345253) D2 D4
    error(0.0345253) D2 D4 ^ D9 D10
    error(0.0345253) D3
    error(0.0345253) D3 D5
    error(0.0345253) D3 D5 ^ D10 D11
    error(0.0345253) D4
    error(0.0345253) D4 D5
    error(0.0345253) D4 D5 ^ D10
    error(0.0345253) D4 ^ D9
    error(0.0345253) D5
    error(0.0345253) D5 ^ D11
    error(0.0345253) D6 D7
    error(0.0345253) D6 D9
    error(0.0345253) D6 D9 ^ D2
    error(0.0345253) D6 L0
    error(0.0345253) D7 D8
    error(0.0345253) D7 D10
    error(0.0345253) D7 L0
    error(0.0345253) D8 D11
    error(0.0345253) D8 D11 ^ D3
    error(0.0

In [109]:
sampler = code.circuit.compile_detector_sampler()
detection_events, observable_flips = sampler.sample(10, separate_observables=True)

In [110]:
m_errors = count_MWPM_logical_errors(code.circuit, nshots)
print(f"There were {m_errors} wrong predictions out of {nshots} shots")

There were 104 wrong predictions out of 1000 shots


In [111]:
num_errors = count_logical_errors(code, nshots)
print(f"There were {num_errors} wrong predictions out of {nshots} shots")

100%|██████████| 1000/1000 [00:06<00:00, 151.39it/s]

There were 47 wrong predictions out of 1000 shots
